# Using Machine Learning to Determine Phenological State Along the Appalachian Trail in Massachusetts

Fall is one of the peak tourist seasons in the Berkshires, as visitors come to see the colorful leaves. Many people time their hikes along the Appalachian Trail around the timing of autumn foliage. This notebook develops models to predict the current phenological (i.e., seasonalal) state of forested pixels along the Appalachian Trail in Massachusetts. This code is run locally, but the best models are stored on GitHub to be used to update a web page on current predictions daily via GitHub Action.

This is a land cover change question, but rather than detecting changes between ecosystem types, these models classify where the canopy is within the seasonal greendown curve. Simpler approaches could be built from specific EVI or NDVI thresholds, the percent change relative to summer or winter values, or rates of change across recent observations. However, these methods are sensitive to noisy satellite data. Smoothing can reduce noise but makes it difficult to track the precise start and end of greendown in near-real time, and logistic greendown curves can only be fit robustly after the season has ended. The machine learning models developed here instead predict, from a snapshot of current and recent conditions, what stage of the greendown curve each pixel is currently experiencing.

This notebook trains and evaluates machine learning models to classify the fall phenological state of forested pixels along the Appalachian Trail in Massachusetts. Each pixel-day observation is assigned one of four labels — **before** (pre-greendown), **early** (early color change), **late** (peak to near-complete greendown), or **after** (post-greendown — based on its day-of-year relative to fitted logistic curve transition dates.

Satellite-derived vegetation indices (EVI and NDVI) from the Harmonized Landsat Sentinel-2 (HLS) dataset are used as input features, along with day length and recent temperature. Two model families are explored:

- **Decision trees** — fast and interpretable, trained on individual observations independently
- **Recurrent neural networks (LSTM)** — trained on temporal sequences of observations per pixel per year, able to exploit within-season context

Multiple variants of each model type are trained to evaluate regularization and training strategies. All models are compared on a held-out test set in the final section.

The best-performing models are deployed via GitHub Actions to generate daily predictions of phenological state within a 50 m buffer of the Appalachian Trail in Massachusetts. Predictions are available at https://k-wheeler.github.io/phenology/. Satellite and weather data are pulled from Google Earth Engine.

This project was developed by Kathryn Wheeler, Ph.D., with the assistance of Claude Code.

In [ ]:
%load_ext autoreload
%autoreload 2

# ── Load libraries and functions ──────────────────────────────────────────────
import os

import ee
import geemap
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio.features
import torch
from pyproj import Transformer as ProjTransformer
from shapely.geometry import shape
from shapely.ops import transform as shapely_transform

from build_data_table import build_feature_table, export_prediction_avg_assets
from constants import DATA_DIR, GREENDOWN_DIR, MODEL_DIR
from decision_trees import evaluate_decision_tree, fit_tree, plot_decision_tree, split_data
from edit_data_table import edit_feature_table
from filter_ci_widths import count_narrow_ci_pixel_years
from fit_greendown_curves import compute_average_transition_dates, compute_transition_dates
from gridmet_utils import fetch_gridmet_cdd_historical
from identify_locations import identify_forests, identify_route_buffer
from model_comparison import _dt_report, _fmt_time, _rnn_report
from plot_feature_distributions import plot_feature_distributions
from read_and_process_hls import compute_hls_indices
from rnn_model import (
    RNN_FEATURE_COLS, RNNPhenologyModel,
    build_rnn_sequences, compute_rnn_norm_stats, normalize_sequences,
    split_sequences, oversample_sequences, train_rnn, evaluate_rnn, save_rnn_model,
    plot_rnn_training_curves, load_rnn_model, _SequenceDataset, collate_rnn_batch, _move_packed
)

ee.Initialize(project='turnkey-lacing-391919')

In [ ]:
# ── Make directories and set spatial-temporal scope ───────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GREENDOWN_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# 100m buffer for training/validation/test data — wider area gives more pixels
# and CI-width diversity for model training.
# Serving (generate_web_outputs.py) stays at the default 50m buffer.
ma_forest    = identify_forests(buffer_m=100)
route_buffer = identify_route_buffer(buffer_m=100)

ma_forest_50    = identify_forests(buffer_m=50)
route_buffer_50 = identify_route_buffer(buffer_m=50)

start_year    = 2013  # First HLS year
download_year = 2025  # Last year to download (includes 2025 for prediction)
train_year    = 2024  # Last year included in model training

refit_models = False # When set to True, this will cause the models to be refit. Available saved models will be loaded when False

## Fit Greendown Curves and Filter by CI Width

For each year and forest pixel within a 100 m buffer of the MA Appalachian Trail, a logistic curve is fit to the HLS EVI time series to estimate three fall transition dates: **start**, **middle**, and **end** of senescence. Cross-year averages of these transition dates are computed for use as features. Pixels where the 95% confidence interval (CI) width of any fitted transition exceeds 15 days are excluded from model training, keeping only pixels with better constrained curve fits.

In [ ]:
# ── Fit logistic curves ───────────────────────────────────────────────────────
all_year_paths = []
for y in range(start_year, download_year + 1):
    print(f'Processing {y}...')
    hls   = compute_hls_indices(route_buffer, ma_forest, y)
    paths = compute_transition_dates(hls, route_buffer, ma_forest, y, data_dir=DATA_DIR, greendown_dir=GREENDOWN_DIR)
    all_year_paths.append(paths)

prev_year_paths = all_year_paths[-1]  # most recent year

# ── Average transition dates across all years ─────────────────────────────────
print('Computing averages...')
avg_paths = compute_average_transition_dates(all_year_paths, data_dir=DATA_DIR, greendown_dir=GREENDOWN_DIR)

# ── Export prediction average assets ──────────────────────────────────────────
# The live prediction (GitHub Action) needs the CI-filtered per-pixel cross-year
# average and the global gap-fill scalar. Re-run on every retrain, then commit:
#   greendown_middle_avg_filtered.tif, greendown_avg_meta.json
print('\nExporting prediction average assets...')
export_prediction_avg_assets(DATA_DIR, GREENDOWN_DIR)

# ── Filter by CI width ────────────────────────────────────────────────────────
# Keep only pixel-years where all three transition CI widths are < 15 days.
print('\nFiltering pixel-years by CI width...')
years = list(range(start_year, download_year + 1))
count_narrow_ci_pixel_years(GREENDOWN_DIR, years)

## Prepare Training Data

Downloads gridMET cold degree-day accumulations for each training year (2013–2024) and builds a flat feature table from HLS satellite observations and the fitted transition dates. Features include EVI and NDVI values and their day-to-day changes, day length, and the pixel's day-of-year relative to the cross-year average transition dates. Each observation is assigned a hard label based on where its day-of-year falls relative to that year's fitted transition dates. Feature distributions are plotted before and after preprocessing, and correlated features are dropped during preprocessing.

In [ ]:
# ── Download gridMET cold degree-days ─────────────────────────────────────────
# Skips any year whose gridmet_cdd_{year}.npz is already cached.
# Must be run before build_feature_table so the cdd_accumulated feature
# can be populated. Re-running is safe — cached years are skipped.
print('\nDownloading gridMET CDD for training years...')
training_years = list(range(start_year, train_year + 1))
for y in training_years:
    fetch_gridmet_cdd_historical(y, route_buffer, DATA_DIR)

# ── Build and edit labeled feature table ──────────────────────────────────────
print('\nBuilding labeled feature table...')
feature_df = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years)

print('\nFeature table before edited')
plot_feature_distributions(feature_df)
# Drop DOY - Avg Start and Avg End DOY: strongly correlated with DOY - Avg Middle DOY

feature_df_edited = edit_feature_table(feature_df, GREENDOWN_DIR, MODEL_DIR)
print('\nNaN counts per column after editing:')
print(feature_df_edited.isna().sum())
plot_feature_distributions(feature_df_edited)

## Decision Tree Models

Three decision tree variants are trained on the preprocessed flat feature table. Data is split 70% training / 30% test with a fixed random seed. If a saved model already exists and `refit_models = False`, the saved model is loaded rather than retraining.

### Variant 1: No Pruning (Baseline)

An unpruned decision tree trained to full depth with no regularization.

In [ ]:
# ── Split data ────────────────────────────────────────────────────────────────
x_train, x_test, y_train, y_test = split_data(feature_df_edited)

# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_no_pruning"
filename = f'decision_tree_model{filename_ext}.joblib'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_tree(x_train, y_train, False)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}.joblib')
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ──────────────────────────────────────────────────────────────────
plot_decision_tree(mdl, x_train, MODEL_DIR)
evaluate_decision_tree(mdl, x_test, y_test, x_train, y_train)

# Without pruning: train accuracy = 1.0 indicates model is overfitting

### Variant 2: Cost-Complexity Pruning

The tree is pruned using cost-complexity pruning (`ccp_alpha` selected by cross-validation in `fit_tree`) to reduce overfitting.

In [ ]:
# ── Split data ────────────────────────────────────────────────────────────────
x_train, x_test, y_train, y_test = split_data(feature_df_edited)

# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_pruned"
filename = f'decision_tree_model{filename_ext}.joblib'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_tree(x_train, y_train, True)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ──────────────────────────────────────────────────────────────────
plot_decision_tree(mdl, x_train, MODEL_DIR)
evaluate_decision_tree(mdl, x_test, y_test, x_train, y_train)

# Training and test accuracies are closer than the unpruned model, indicating reduced overfitting

### Variant 3: Feature Selection

The three spectral delta features (`ndvi_delta`, `ndvi_delta2`, `evi_delta`) are removed before training, retaining only the features that contribute most to the unpruned tree.

In [ ]:
# ── Prepare and split data ────────────────────────────────────────────────────
feature_df_edited_select = feature_df_edited.drop(columns=['ndvi_delta2', 'ndvi_delta', 'evi_delta'])

x_train, x_test, y_train, y_test = split_data(feature_df_edited_select)

# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_select_features"
filename = f'decision_tree_model{filename_ext}.joblib'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    mdl, training_time_sec = fit_tree(x_train, y_train, True)
    joblib.dump(mdl, os.path.join(MODEL_DIR, filename))
    print(f'Model saved to {MODEL_DIR}/{filename}')
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt'), 'w') as f:
        f.write(f'{training_time_sec:.2f} seconds\n')
else:
    mdl = joblib.load(os.path.join(MODEL_DIR, filename))
    with open(os.path.join(MODEL_DIR, f'training_time{filename_ext}.txt')) as f:
        training_time_sec = float(f.read().split()[0])

# ── Evaluate ──────────────────────────────────────────────────────────────────
plot_decision_tree(mdl, x_train, MODEL_DIR)
evaluate_decision_tree(mdl, x_test, y_test, x_train, y_train)

# Improved test accuracy relative to the pruned full-feature model

## Recurrent Neural Network (LSTM) Models

A 2-layer LSTM is trained on per-pixel-year sequences of HLS satellite observations. Unlike the decision tree, the LSTM can use the temporal context of prior observations within a season. The feature table is rebuilt with pixel IDs retained so observations can be grouped into sequences; temperature features are also included. Sequences are split 70% train / 15% validation / 15% test by pixel-year to prevent data leakage across splits. The same base architecture is used across all variants; they differ in regularization and training strategy.

### 50m Buffer Variants — Data Preparation

All model variants for decision tree and RNN were originally trained, validated, and tested using pixels within 50 m of the Appalachian Trail, the same spatial boundary we want to predict. The RNNs were not performing well so the training and model selection data set was expanded to include pixels within 100 m of the Appalachian Trail. Most of the model variants are shown with the 100 m buffer, but below are two of the model variants fir to the 50 m buffer data. 

In [ ]:
# ── Build 50m spatial mask ────────────────────────────────────────────────────

# Single GEE call to retrieve the 50m buffer geometry coordinates
buffer_geojson = route_buffer_50.getInfo()
geom_wgs84 = shape(buffer_geojson)

# Use any greendown GeoTIFF as the grid reference (same CRS/extent as CI width arrays)
ref_year = training_years[0]
ref_path = os.path.join(GREENDOWN_DIR, f'greendown_start_ci_width_{ref_year}.tif')
with rasterio.open(ref_path) as _src:
    _raster_transform = _src.transform
    _raster_shape     = (_src.height, _src.width)
    _raster_crs       = _src.crs

proj_to_raster = ProjTransformer.from_crs('EPSG:4326', _raster_crs, always_xy=True)
geom_proj = shapely_transform(proj_to_raster.transform, geom_wgs84)

spatial_mask_50m = rasterio.features.rasterize(
    [(geom_proj, 1)],
    out_shape=_raster_shape,
    transform=_raster_transform,
    fill=0,
    dtype='uint8'
).astype(bool)
print(f'50m mask: {spatial_mask_50m.sum()} of {spatial_mask_50m.size} pixels')

# ── Build RNN feature table (50m buffer) ──────────────────────────────────────
print('Building RNN feature table (50m buffer)...')
rnn_df_50m = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years,
                                  retain_pixel_id=True, include_temperature=True,
                                  spatial_mask=spatial_mask_50m)
n_pixels_50m = rnn_df_50m['pixel_id'].nunique()
print(f'  {len(rnn_df_50m)} observations across {n_pixels_50m} unique pixels')

# ── Build sequences ───────────────────────────────────────────────────────────
sequences_50m = build_rnn_sequences(rnn_df_50m)
seq_lens_50m  = [len(s[0]) for s in sequences_50m]
print(f'  {len(sequences_50m)} sequences  |  '
      f'length range: {min(seq_lens_50m)}–{max(seq_lens_50m)}  |  '
      f'median: {int(sorted(seq_lens_50m)[len(seq_lens_50m)//2])}')

# ── Normalize ─────────────────────────────────────────────────────────────────
norm_stats_50m = compute_rnn_norm_stats(sequences_50m)
normalize_sequences(sequences_50m, norm_stats_50m)
nan_check_50m = sum(np.isnan(f).any() for f, _ in sequences_50m)
print(f'  Sequences with NaN after normalization: {nan_check_50m}  (must be 0)')

# ── Split ─────────────────────────────────────────────────────────────────────
trainval_50m, test_50m = split_sequences(sequences_50m,  val_frac=0.15,  seed=42)
train_50m,    val_50m  = split_sequences(trainval_50m,   val_frac=0.176, seed=42)
print(f'  Train: {len(train_50m)}  |  Val: {len(val_50m)}  |  Test: {len(test_50m)} sequences')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Variant 1: Dropout Only (50m Buffer) - Baseline

Baseline LSTM with dropout regularization (rate 0.3) trained on pixels within the 50 m AT buffer. Hyperparameters are identical to Variant 3 (100 m baseline) to isolate the effect of spatial extent.

In [ ]:
# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_dropout_only_50m"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_50m, val_50m,
                  epochs=60, lr=1e-3, batch_size=64, device=device)
    save_rnn_model(model, norm_stats_50m, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats_50m, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_50m, device=device)

print('\nTest results:')
evaluate_rnn(model, test_50m, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_50m, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

### Variant 2: L1 Regularization (50m Buffer)

L1 regularization with early stopping trained on pixels within the 50 m AT buffer. Hyperparameters are identical to Variant 6 (100 m L1) to isolate the effect of spatial extent.

In [ ]:
# L1 regularization penalizes large weights; chosen because decision tree
# feature importances suggest several features have low predictive value.
# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_L1R_50m"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_50m, val_50m, epochs=100, lr=1e-3,
                                                  batch_size=64, device='cpu', early_stopping=True,
                                                  patience=10, L1_regular=True, l1_lambda=1e-4)
    save_rnn_model(model, norm_stats_50m, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats_50m, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_50m, device=device)

print('\nTest results:')
evaluate_rnn(model, test_50m, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_50m, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

### Data Preparation for RNN (100m)

In [ ]:
# ── Build RNN feature table ───────────────────────────────────────────────────
# Rebuilds the feature table with pixel_id retained and temperature features
# enabled. Reuses already-downloaded HLS stacks and gridMET cache — no new GEE calls.
print('Building RNN feature table (with pixel_id + temperature)...')
rnn_df = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years,
                              retain_pixel_id=True,
                              include_temperature=True)
n_pixels = rnn_df['pixel_id'].nunique()
print(f'  {len(rnn_df)} observations across {n_pixels} unique pixels')
print(f'  NaN counts:\n{rnn_df[RNN_FEATURE_COLS].isna().sum().to_string()}')

# ── Build sequences ───────────────────────────────────────────────────────────
# Group observations by pixel-year into temporal sequences (one per pixel per year).
sequences = build_rnn_sequences(rnn_df)
seq_lens  = [len(s[0]) for s in sequences]
print(f'\n  {len(sequences)} sequences  |  '
      f'length range: {min(seq_lens)}–{max(seq_lens)}  |  '
      f'median: {int(sorted(seq_lens)[len(seq_lens)//2])}')

# ── Normalize ─────────────────────────────────────────────────────────────────
# normalize_sequences replaces any remaining NaN with 0.0 (= feature mean
# in z-score space), so NaN in temperature features cannot reach the LSTM.
norm_stats = compute_rnn_norm_stats(sequences)
normalize_sequences(sequences, norm_stats)
nan_check = sum(np.isnan(f).any() for f, _ in sequences)
print(f'  Sequences with NaN after normalization: {nan_check}  (must be 0)')

# ── Split ─────────────────────────────────────────────────────────────────────
# Three-way split: 70% train / 15% val / 15% test.
# Test set is held out entirely until final evaluation — val drives early stopping.
# split_sequences splits by pixel-year so no pixel leaks across splits.
trainval_seqs, test_seqs  = split_sequences(sequences,     val_frac=0.15, seed=42)
train_seqs,    val_seqs   = split_sequences(trainval_seqs, val_frac=0.176, seed=42)
# 0.176 × 85% ≈ 15% of total
print(f'  Train: {len(train_seqs)}  |  Val: {len(val_seqs)}  |  Test: {len(test_seqs)} sequences')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Variant 3: Dropout Only

Baseline LSTM with dropout regularization (rate 0.3). Trained for 60 epochs with no early stopping or class balancing.

In [ ]:
# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_dropout_only"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_seqs, val_seqs,
                  epochs=60, lr=1e-3, batch_size=64, device=device)
    save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_seqs, device=device)

print('\nTest results:')
evaluate_rnn(model, test_seqs, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_seqs, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

# Performance varies widely across classes; early and late states have the
# smallest share of observations and show the lowest precision

### Variant 4: Class Balancing

Training sequences are oversampled so all four phenological classes are equally represented. This addresses class imbalance, as "before" and "after" states dominate the dataset relative to the transitional "early" and "late" states.

In [ ]:
# Oversample minority-class sequences in the training set only.
# target_ratio=1.0 fully balances all classes to the majority count.
# Lower (e.g. 0.5) if you see overfitting on the val classification report.
train_seqs_rebalanced = oversample_sequences(train_seqs, target_ratio=1.0)
print(f'  After oversampling: {len(train_seqs_rebalanced)} training sequences')

# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_class_balancing"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_seqs_rebalanced, val_seqs,
                  epochs=60, lr=1e-3, batch_size=64, device=device)
    save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext)
    save_rnn_model(model, norm_stats, MODEL_DIR, "")  # save as current best RNN model
else:
    model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_seqs, device=device)

print('\nTest results:')
evaluate_rnn(model, test_seqs, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_seqs_rebalanced, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

# Class rebalancing improved early/late precision minimally; the high train
# accuracy with diverging val loss indicates the model is overfitting

### Variant 5: Early Stopping

Training halts when validation loss stops improving (patience = 10 epochs), reducing training time and limiting overfitting to the training set.

In [ ]:
# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_early_stopping"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_seqs_rebalanced, val_seqs,
        epochs=60, lr=1e-3, batch_size=64, device=device,
        early_stopping=True, patience=10)
    save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_seqs, device=device)

print('\nTest results:')
evaluate_rnn(model, test_seqs, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_seqs_rebalanced, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

### Variant 6: L1 Regularization

L1 (lasso) regularization is added to the loss function to penalize large weights and encourage sparsity across the network, analogous to feature selection in the decision tree.

In [ ]:
# L1 regularization penalizes large weights; chosen because decision tree
# feature importances suggest several features have low predictive value.
# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_L1R"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_seqs, val_seqs, epochs=100, lr=1e-3,
                                                  batch_size=64, device='cpu', early_stopping=True,
                                                  patience=10, L1_regular=True, l1_lambda=1e-4)
    save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('\nValidation results:')
evaluate_rnn(model, val_seqs, device=device)

print('\nTest results:')
evaluate_rnn(model, test_seqs, device=device)

print('\nTrain results (compare to test to check overfitting):')
evaluate_rnn(model, train_seqs_rebalanced, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

### Variant 7: Soft Labels

Instead of hard (one-hot) labels, each observation is assigned a probability distribution over the four classes based on its distance from each transition date relative to the fitted CI width. Observations near a boundary receive probability mass on both adjacent classes; observations far from any boundary receive near-hard labels. Training uses soft cross-entropy loss. Evaluation uses hard labels as the ground truth.

In [ ]:
rnn_df = build_feature_table(DATA_DIR, GREENDOWN_DIR, training_years,
                              retain_pixel_id=True, include_temperature=True)
sequences = build_rnn_sequences(rnn_df, use_soft_labels=True)

seq_lens  = [len(s[0]) for s in sequences]
print(f'\n  {len(sequences)} sequences  |  '
      f'length range: {min(seq_lens)}–{max(seq_lens)}  |  '
      f'median: {int(sorted(seq_lens)[len(seq_lens)//2])}')

norm_stats = compute_rnn_norm_stats(sequences)
normalize_sequences(sequences, norm_stats)
nan_check = sum(np.isnan(f).any() for f, _ in sequences)
print(f'  Sequences with NaN after normalization: {nan_check}  (must be 0)')

trainval_seqs, test_seqs  = split_sequences(sequences,     val_frac=0.15, seed=42)
train_seqs,    val_seqs   = split_sequences(trainval_seqs, val_frac=0.176, seed=42)
# 0.176 × 85% ≈ 15% of total
print(f'  Train: {len(train_seqs)}  |  Val: {len(val_seqs)}  |  Test: {len(test_seqs)} sequences')

train_seqs_rebalanced = oversample_sequences(train_seqs, target_ratio=1.0)
print(f'  After oversampling: {len(train_seqs_rebalanced)} training sequences')

# ── Variant filename and extension ────────────────────────────────────────────
filename_ext = "_soft_labels"
filename = f'rnn_model{filename_ext}.pt'

# ── Fit or load model ─────────────────────────────────────────────────────────
if refit_models or not os.path.exists(os.path.join(MODEL_DIR, filename)):
    model = RNNPhenologyModel(input_size=len(RNN_FEATURE_COLS), hidden_size=64, num_layers=2)
    model, history, training_time_sec = train_rnn(model, train_seqs, val_seqs, epochs=100, lr=1e-3,
                                                  batch_size=64, device='cpu', early_stopping=True,
                                                  patience=10, L1_regular=True, l1_lambda=1e-4)
    save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext)
else:
    model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
    training_time_sec = history['training_time_sec']

# ── Evaluate against hard labels ──────────────────────────────────────────────
# Soft label argmax distorts per-class metrics: "early" observations near the
# start boundary have argmax="before" in their soft distribution, so the model
# gets no credit for correct "early" predictions → precision appears as 0.
# Hard labels are the correct ground truth for evaluation; soft labels are only
# a training signal. Same norm_stats and split seeds guarantee identical splits.
sequences_hard = build_rnn_sequences(rnn_df, use_soft_labels=False)
normalize_sequences(sequences_hard, norm_stats)
trainval_hard, test_hard = split_sequences(sequences_hard,  val_frac=0.15,  seed=42)
train_hard,    val_hard  = split_sequences(trainval_hard,   val_frac=0.176, seed=42)
train_hard_rebalanced = oversample_sequences(train_hard, target_ratio=1.0)

print('\nValidation results (hard labels):')
evaluate_rnn(model, val_hard, device=device)

print('\nTest results (hard labels):')
evaluate_rnn(model, test_hard, device=device)

print('\nTrain results (hard labels, compare to test to check overfitting):')
evaluate_rnn(model, train_hard_rebalanced, device=device)

print('\nTraining time (seconds):')
print(training_time_sec)

plot_rnn_training_curves(history, filename_ext, MODEL_DIR)

## Model Comparison

All trained models are loaded from disk and evaluated on the same held-out test set. The summary table is sorted by test accuracy. Metrics shown include overall accuracy, per-class precision, and training time for both training and test data.

In [ ]:
# ── Split data ────────────────────────────────────────────────────────────────
_x_tr_full, _x_te_full, _y_tr_full, _y_te_full = split_data(feature_df_edited)
_x_tr_sel,  _x_te_sel,  _y_tr_sel,  _y_te_sel  = split_data(feature_df_edited_select)

_seqs_h = build_rnn_sequences(rnn_df, use_soft_labels=False)
_norm_h  = compute_rnn_norm_stats(_seqs_h)
normalize_sequences(_seqs_h, _norm_h)
_trainval_h, _test_h = split_sequences(_seqs_h,      val_frac=0.15,  seed=42)
_train_h,    _val_h  = split_sequences(_trainval_h,  val_frac=0.176, seed=42)

if 'rnn_df_50m' in dir():
    _seqs_50m = build_rnn_sequences(rnn_df_50m, use_soft_labels=False)
    _norm_50m  = compute_rnn_norm_stats(_seqs_50m)
    normalize_sequences(_seqs_50m, _norm_50m)
    _trainval_50m, _test_50m = split_sequences(_seqs_50m, val_frac=0.15, seed=42)
    _train_50m,    _val_50m  = split_sequences(_trainval_50m, val_frac=0.176, seed=42)

# ── Collect metrics ───────────────────────────────────────────────────────────
rows = []
LABEL_ORDER = ['before', 'early', 'late', 'after']

dt_configs = [
    ('_no_pruning',      'DT No Pruning',      _x_tr_full, _y_tr_full, _x_te_full, _y_te_full),
    ('_pruned',          'DT Pruned',           _x_tr_full, _y_tr_full, _x_te_full, _y_te_full),
    ('_select_features', 'DT Select Features',  _x_tr_sel,  _y_tr_sel,  _x_te_sel,  _y_te_sel),
]
for ext, name, x_tr, y_tr, x_te, y_te in dt_configs:
    path = os.path.join(MODEL_DIR, f'decision_tree_model{ext}.joblib')
    if not os.path.exists(path):
        print(f'  Skipping {name} — file not found'); continue
    m = joblib.load(path)
    time_path = os.path.join(MODEL_DIR, f'training_time{ext}.txt')
    t = float(open(time_path).read().split()[0]) if os.path.exists(time_path) else float('nan')
    tr, te = _dt_report(m, x_tr, y_tr, x_te, y_te)
    row = {'Model': name, 'Type': 'Decision Tree', 'Training Time (s)': t,
           'Train Accuracy': tr['accuracy'], 'Test Accuracy': te['accuracy']}
    for cls in LABEL_ORDER:
        row[f'Train Prec ({cls})'] = tr[cls]['precision']
        row[f'Test Prec ({cls})']  = te[cls]['precision']
    rows.append(row)

rnn_configs = [
    ('_dropout_only',    'RNN Dropout'),
    ('_class_balancing', 'RNN Class Balance'),
    ('_early_stopping',  'RNN Early Stop'),
    ('_L1R',             'RNN L1'),
    ('_soft_labels',     'RNN Soft Labels'),
]
for ext, name in rnn_configs:
    if not os.path.exists(os.path.join(MODEL_DIR, f'rnn_model{ext}.pt')):
        print(f'  Skipping {name} — file not found'); continue
    m, _, hist = load_rnn_model(MODEL_DIR, ext)
    t = hist.get('training_time_sec', float('nan')) if hist else float('nan')
    tr = _rnn_report(m, _train_h)
    te = _rnn_report(m, _test_h)
    row = {'Model': name, 'Type': 'RNN', 'Training Time (s)': t,
           'Train Accuracy': tr['accuracy'], 'Test Accuracy': te['accuracy']}
    for cls in LABEL_ORDER:
        row[f'Train Prec ({cls})'] = tr[cls]['precision']
        row[f'Test Prec ({cls})']  = te[cls]['precision']
    rows.append(row)

if 'rnn_df_50m' in dir():
    rnn_50m_configs = [
        ('_dropout_only_50m', 'RNN Dropout 50m'),
        ('_L1R_50m',          'RNN L1 50m'),
    ]
    for ext, name in rnn_50m_configs:
        if not os.path.exists(os.path.join(MODEL_DIR, f'rnn_model{ext}.pt')):
            print(f'  Skipping {name} — file not found'); continue
        m, _, hist = load_rnn_model(MODEL_DIR, ext)
        t = hist.get('training_time_sec', float('nan')) if hist else float('nan')
        tr = _rnn_report(m, _train_50m)
        te = _rnn_report(m, _test_50m)
        row = {'Model': name, 'Type': 'RNN 50m', 'Training Time (s)': t,
               'Train Accuracy': tr['accuracy'], 'Test Accuracy': te['accuracy']}
        for cls in LABEL_ORDER:
            row[f'Train Prec ({cls})'] = tr[cls]['precision']
            row[f'Test Prec ({cls})']  = te[cls]['precision']
        rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
df_sorted = df.sort_values('Test Accuracy', ascending=False)

def _fmt_time(s):
    if np.isnan(s):
        return 'N/A'
    s = int(s)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}'

# ── Tables ────────────────────────────────────────────────────────────────────
summary = df_sorted[['Type', 'Training Time (s)', 'Train Accuracy', 'Test Accuracy']].copy()
summary.insert(2, 'Training Time', summary['Training Time (s)'].apply(_fmt_time))
summary = summary.drop(columns='Training Time (s)')
print('=== Summary (sorted by test accuracy) ===')
print(summary.to_string(float_format='%.4f'))

print('\n=== Test Precision by Class ===')
print(df_sorted[[f'Test Prec ({c})' for c in LABEL_ORDER]]
      .rename(columns={f'Test Prec ({c})': c for c in LABEL_ORDER})
      .to_string(float_format='%.4f'))

print('\n=== Train Precision by Class ===')
print(df_sorted[[f'Train Prec ({c})' for c in LABEL_ORDER]]
      .rename(columns={f'Train Prec ({c})': c for c in LABEL_ORDER})
      .to_string(float_format='%.4f'))

# ── Figures ───────────────────────────────────────────────────────────────────
models = df_sorted.index.tolist()
x      = np.arange(len(models))
w      = 0.35
colors = ['steelblue', 'darkorange', 'forestgreen', 'firebrick']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training time
axes[0, 0].bar(x, df_sorted['Training Time (s)'])
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models, rotation=30, ha='right')
axes[0, 0].set_title('Training Time')
axes[0, 0].set_ylabel('Seconds')

# Train vs test accuracy
axes[0, 1].bar(x - w/2, df_sorted['Train Accuracy'], width=w, label='Train')
axes[0, 1].bar(x + w/2, df_sorted['Test Accuracy'],  width=w, label='Test')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models, rotation=30, ha='right')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()

# Test precision by class
for i, cls in enumerate(LABEL_ORDER):
    axes[1, 0].plot(models, df_sorted[f'Test Prec ({cls})'],
                    marker='o', label=cls, color=colors[i])
axes[1, 0].set_xticks(range(len(models)))
axes[1, 0].set_xticklabels(models, rotation=30, ha='right')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_title('Test Precision by Class')
axes[1, 0].legend()

# Train precision by class
for i, cls in enumerate(LABEL_ORDER):
    axes[1, 1].plot(models, df_sorted[f'Train Prec ({cls})'],
                    marker='o', label=cls, color=colors[i])
axes[1, 1].set_xticks(range(len(models)))
axes[1, 1].set_xticklabels(models, rotation=30, ha='right')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_title('Train Precision by Class')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


# Conclusions

## Model Performance
These analyses were initially run using only pixels within 50 m of the Appalachian Trail for training and testing, then expanded to 100 m after model performance proved unsatisfactory.

Overall accuracy (the fraction of all observations classified correctly) is a misleading metric here because correctly identifying the transitional classes (early and late greendown) matters more than classifying the more common before and after states. The canopy spends more time in the before and after states than during active senescence (i.e., color change), so the early and late classes are inherently underrepresented in the training data. Precision(the fraction of predictions for a given class that are actually correct) for these classes is therefore a more informative metric for evaluating performance on these transitional classes.

When trained on 50 m buffer data, the recurrent neural network (RNN, a model that processes observations as a time series drawing on the history of prior observations within a season) with dropout regularization (a technique that randomly disables a subset of model connections during each training step to prevent the model from over-relying on any single feature) achieved the best overall test accuracy, but had poor precision for the early (0.60) and late (0.70) greendown classes. Adding class balancing (oversampling the underrepresented early and late training sequences so all classes are equally represented), early stopping (halting training once validation performance stops improving), or L1 regularization (adding a penalty on large model weights to favor simpler solutions) did not improve accuracy or precision for these transitional classes. Expanding training data to 100 m substantially improved model performance: the 100 m dropout-only RNN achieved the highest overall accuracy, and none of the additional regularization strategies improved upon it, though early stopping did meaningfully reduce training time at the detriment of model performance.

Among the decision trees (models that classify observations by applying a series of yes/no thresholds on input features), the unpruned full-feature model had the highest training accuracy. However, a training accuracy of 1.0 indicates severe overfitting: the model has memorized the training data rather than learned patterns that generalize to new observations and years. The feature-selected model, which also applied cost-complexity pruning to limit tree depth, achieved the best test accuracy, with a similar degree of overfitting to the pruning-only model. Removing the three lowest-importance spectral delta features slightly reduced precision for the early greendown class compared to pruning alone, but probably not enough to warrant retaining them given the additional computational cost (slower training and larger stored data tables). The unpruned full-feature model achieved the highest precision for the early and late greendown classes, but its overfitting makes it unlikely to generalize well to future years.

All decision tree variants had lower overall test accuracy than the RNN models but notably higher precision for the early and late greendown classes. Precision for these transitional classes declined in both model types when the training footprint expanded from 50 m to 100 m. This is consistent with the general observation that decision trees tend to outperform neural network models on small datasets, while the latter benefit more from additional data to learn robust temporal patterns.

For operational greendown state predictions, the feature-selected pruned decision tree and the 100 m dropout-only RNN will be used. Future improvements could include training the RNN on more pixels and incorporating higher-level spectral features such as satellite embeddings.

## Claude Code Experience
Strengths
* Claude could quickly build in data checks to confirm things are working correctly. It allowed me to complete this analysis and build a web page from scratch much faster than I could have done otherwise.
* Claude could run most of the work using a lower-effort model (Sonnet 4.6), but when issues arose that Sonnet could not resolve, switching to the higher-effort model (Opus 4.8) was helpful. Opus 4.8 was also able to find bugs in Sonnet's work without being prompted.
* Claude flagged actions that would be computationally expensive or cause storage issues. For example, when I laid out my plan to run a daily prediction via a GitHub Action, it pointed out that storing the required large data files (~2–3 GB) on GitHub would not work. I described how to split the workflow and file storage between local and GitHub. This was obvious to me, but someone less experienced with allocating computation and storage might not have thought to push back.
* I enjoyed the planning feature and boucing ideas off of it, which felt like I was collaborating with a colleague sometimes. 
* Claude would sometimes push back on my suggestions or questions, which I appreciated. For example, I was interested in using GEFS meteorological forecast data because the first time step in a forecast represents current conditions, which I had used in my PhD work to get meteorological data at low latency with uncertainty estimates. Claude questioned why I wanted a forecast product for what seemed like a hindcast application. Ultimately I used gridMET because it has finer spatial resolution than GEFS, but GEFS could have been a useful workaround that Claude would not have known to suggest.
* Claude serves as a great learning tool, with the ability to ask questions at any point, especially about how things work and token usage.

Weaknesses
* Context window memory proved to be a bigger challenge than I initially expected. The conversation had to be compressed regularly, and Claude did not reliably remember key details I assumed it would retain. In the future I would rely more heavily on reference files that document key decisions and project state.
* Debugging could be more challenging because I did not have the same level of ownership and understanding of the code that I would have had if I had written it myself.
* This workflow required setting up several permissions on Google Earth Engine and GitHub that I had to handle myself. Claude walked me through most of it, but I still needed enough background knowledge to carry out the steps. There was also a fair amount of trial and error to get everything working correctly (missed permissions, or permissions set up incorrectly based on Claude's guidance).
* Sometimes Claude did not fully implement changes, and the scope of what it changed varied. Sometimes it would only update a helper function without touching the main script. Sometimes it would automatically commit and push changes; other times it would not, and I would have to. These are easy to catch but require staying aware.
* Sometimes Claude would inadvertently undo edits I had made when asked to change something else. For example, I had added a call to plot feature distributions before the data-editing step so I could see the raw features. Whenever I asked Claude to modify the editing function, it would move the plotting call to after the editing step and change the variable from `feature_df` to `feature_df_edited`. I had to keep reverting the change, and even after explicitly asking Claude to stop, the behavior recurred after context compression. This highlights the importance of version control and reviewing all file changes.
* Some tasks took far more time and tokens than expected. For example, reorganizing outputs into separate subdirectories (which seemed straightforward) required Claude to determine which output belonged in which folder. The situation was made worse when I switched to a lower-effort model (Haiku) partway through. Haiku broke the code and reverted all changes by checking out a previous commit, forcing a restart from scratch. The main root causes were: (1) the Jupyter notebook still had outputs saved, so Claude consumed thousands of tokens reading cell outputs it did not need; and (2) NotebookEdit re-emits the entire cell source on every call, making edits to large code cells expensive. After that experience, I started saving the notebook without outputs before asking Claude to edit it. I also began writing smaller code chunks in the notebook, keeping larger or frequently edited logic in standalone Python files so Claude could work on them more efficiently. Saving outputs from long-running steps (such as model training) to separate files rather than keeping them in the notebook is also important.
* Sometimes Claude simply did not follow the instruction given. For example, I asked it to place a new function in a specific file and provided the filename, but Claude saved it under a different name.
* When asked to make formatting consistent (such as docstrings on functions), Claude noted that only about half the functions it had written had docstrings, and those that did used inconsistent styles. In the future I would include style guidelines in a reference file from the start.
* It does not know everything, and the limits of its knowledge became apparent in areas where I have deep expertise. For example, the spectral delta features at the beginning of each pixel-year sequence had missing values (NAs) because there were no prior observations to calculate a change from. Claude's default suggestion was to fill these with the column mean, which is a reasonable general approach for randomly missing data but inappropriate here because these values should be toward the high end of the range (reflecting early-season conditions). These rows represented a small fraction of records and should be (and were) dropped instead.
* Claude struggled with ensuring that spatial data from different sources were properly aligned and shared the same coordinate reference system (CRS). When working with spatial data, I always hard-code checks of the CRS. In the future I will include in a reference file what CRS I want and specify that it should be checked each time new spatial data is added. At one point some data had been projected into the Louisiana State Plane system, which should have been flagged immediately given that all analyses were in Massachusetts.

Many common coding practices also apply:
* Think about the big picture but build incrementally to debug and quality-check as you go.
* Work with a smaller subset of data before expanding. Downloading new data took considerable time, and when the analysis changed, some data had to be re-downloaded.
* Commit and push frequently, because Claude did occasionally break things and files sometimes needed to be reverted.
* Modularize code using separate files for distinct components. Claude naturally creates helper functions but did not always place them in separate files without being prompted.

# Efficiency Choices
* Smoothed the Appalachian Trail geometry to reduce complexity.
* More computationally expensive steps were run locally once and their outputs stored:
    * Pulling data from Google Earth Engine, storing as processed data locally, and uploading historical average GeoTIFFs to GitHub.
    * Training models locally and then uploading them to GitHub.
* GitHub Actions pull the most recent observation values and run the prediction.

In [ ]:
# ── Save best decision tree for GitHub ────────────────────────────────────────
filename_ext = "_select_features"
filename = f'decision_tree_model{filename_ext}.joblib'

mdl = joblib.load(os.path.join(MODEL_DIR, filename))

filename_best = 'decision_tree_model.joblib'
joblib.dump(mdl, os.path.join(MODEL_DIR, filename_best))

# ── Save best RNN model for GitHub ──────────────────────────────────────────── 
filename_ext = "_dropout_only"
filename = f'rnn_model{filename_ext}.pt'

model, norm_stats, history = load_rnn_model(MODEL_DIR, filename_ext)
save_rnn_model(model, norm_stats, MODEL_DIR, history, training_time_sec, filename_ext="")
